# 🚀 Amazing-QR: Tam Otomatik (CSV) QR Oluşturucu

Bu notebook, **Amazing-QR** projesini tamamen CSV (sipariş listesi) odaklı çalıştırmak için optimize edilmiştir.

---

## 🛠️ 1. Kurulum ve Hazırlık


In [ ]:
# Projeyi klonla (Eğer içeride değilsek)
import os
if not os.path.exists('amzqr'):
    !git clone https://github.com/orioninsist/amazing-qr.git
    %cd amazing-qr

# Gereksinimleri yükle
!pip install -q pandas ipywidgets amzqr
!pip install -e . # amzqr paketini sisteme yükle

# Klasör yapısını hazırla
!mkdir -p inputs/assets
!mkdir -p output


## 📊 2. Sipariş Listesini Görüntüle


In [ ]:
import pandas as pd
import os
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab import files

csv_path = 'inputs/order.csv'
os.makedirs('inputs/assets', exist_ok=True)

def upload_assets(b):
    display(HTML("<div style='color: #2980b9;'>📁 Lütfen arka plan resimlerini veya GIF dosyalarını seçin...</div>"))
    uploaded = files.upload()
    for filename in uploaded.keys():
        dest = os.path.join('inputs/assets', filename)
        with open(dest, 'wb') as f:
            f.write(uploaded[filename])
        display(HTML(f"<div style='color: #27ae60;'>✅ {filename} yüklendi.</div>"))

display(HTML("<h2 style='color: #2c3e50;'>🖼️ 2. Arka Plan / Logo Yükle (Opsiyonel)</h2>"))
upload_btn = widgets.Button(description="📁 Dosya Yükle", button_style='info', layout={'width': '200px', 'height': '40px'})
upload_btn.on_click(upload_assets)
display(upload_btn)

if not os.path.exists(csv_path):
    df_empty = pd.DataFrame(columns=['words', 'version', 'level', 'picture', 'colorized', 'contrast', 'brightness', 'save_name'])
    df_empty.to_csv(csv_path, index=False)
    display(HTML("<div style='padding: 20px; background: #e3f2fd; border-left: 5px solid #2196f3; border-radius: 4px;'>ℹ️ inputs/order.csv bulunamadı, yeni bir taslak oluşturuldu.</div>"))

try:
    df = pd.read_csv(csv_path)
    display(HTML("<h3 style='color: #2c3e50;'>📋 Mevcut Sipariş Listesi</h3>"))
    display(df)
except Exception as e:
    display(HTML(f"<div style='padding: 20px; background: #ffebee; border-left: 5px solid #f44336; border-radius: 4px;'>❌ CSV okuma hatası: {e}</div>"))


## ✍️ 3. Siparişleri Düzenle (Editor)


In [ ]:
import pandas as pd
import os
import re
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from google.colab import files

csv_path = 'inputs/order.csv'

def upload_csv(b):
    display(HTML("<div style='color: #2980b9;'>📁 Lütfen order.csv dosyasını seçin...</div>"))
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        with open(csv_path, 'wb') as f:
            f.write(uploaded[filename])
        display(HTML(f"<div style='color: #27ae60;'>✅ {filename} yüklendi ve inputs/order.csv olarak kaydedildi.</div>"))
        refresh_view()

def slugify(value):
    value = re.sub(r'^https?://', '', str(value))
    value = re.sub(r'[^\w\s-]', '_', value).strip().lower()
    return re.sub(r'[-\s]+', '_', value)[:50]

def load_data():
    if not os.path.exists(csv_path):
        return pd.DataFrame(columns=['selected', 'words', 'version', 'level', 'picture', 'colorized', 'contrast', 'brightness', 'save_name'])
    try:
        df = pd.read_csv(csv_path)
    except:
        return pd.DataFrame(columns=['selected', 'words', 'version', 'level', 'picture', 'colorized', 'contrast', 'brightness', 'save_name'])
    cols = ['selected', 'words', 'version', 'level', 'picture', 'colorized', 'contrast', 'brightness', 'save_name']
    for c in cols:
        if c not in df.columns:
            if c == 'selected': df[c] = True
            else: df[c] = None
    df['selected'] = df['selected'].apply(lambda x: str(x).lower() == 'true' if not isinstance(x, bool) else x)
    df['version'] = pd.to_numeric(df['version'], errors='coerce').fillna(1).astype(int)
    df['level'] = df['level'].fillna('H')
    df['contrast'] = pd.to_numeric(df['contrast'], errors='coerce').fillna(1.0).astype(float)
    df['brightness'] = pd.to_numeric(df['brightness'], errors='coerce').fillna(1.0).astype(float)
    for i, row in df.iterrows():
        if pd.isna(row['colorized']) or str(row['colorized']).strip() in ['', 'nan', 'None']:
            df.at[i, 'colorized'] = True if pd.notna(row.get('picture')) and str(row.get('picture')).strip() not in ['', 'nan', 'None'] else False
        else:
            df.at[i, 'colorized'] = str(row['colorized']).lower() == 'true'
        if pd.isna(row['save_name']) or str(row['save_name']).strip() in ['', 'nan', 'None']:
            ext = '.gif' if str(row.get('picture', '')).lower().endswith('.gif') else '.png'
            slug = slugify(row['words']) or f"qr_{i+1}"
            df.at[i, 'save_name'] = f"{slug}{ext}"
    return df[cols]

def save_data(df):
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    df.to_csv(csv_path, index=False)

output = widgets.Output()

def refresh_view():
    with output:
        clear_output()
        df = load_data()
        
        up_csv_btn = widgets.Button(description="📁 CSV Yükle", button_style='info', layout={'width': '150px', 'height': '35px'})
        add_btn_top = widgets.Button(description="➕ Yeni Sipariş", button_style='success', layout={'width': '150px', 'height': '35px'})
        bulk_btn_top = widgets.Button(description="🛠️ Toplu Düzenle", button_style='warning', layout={'width': '150px', 'height': '35px'})
        
        def toggle_all(b):
            df_local = load_data()
            all_selected = df_local['selected'].all()
            df_local['selected'] = not all_selected
            save_data(df_local)
            refresh_view()
            
        toggle_btn = widgets.Button(description="✅ Hepsini Seç/Kaldır", button_style='primary', layout={'width': '180px', 'height': '35px'})
        toggle_btn.on_click(toggle_all)
        
        up_csv_btn.on_click(upload_csv)
        add_btn_top.on_click(lambda b: show_editor(None))
        bulk_btn_top.on_click(lambda b: show_bulk_editor())
        
        display(widgets.HBox([up_csv_btn, add_btn_top, bulk_btn_top, toggle_btn], layout={'margin': '0 0 15px 0', 'gap': '10px'}))

        if df.empty:
            display(HTML("<div style='padding: 20px; background: #fff4e5; border-left: 5px solid #ffa117; border-radius: 4px; margin: 10px 0;'>⚠️ Sipariş listesi boş.</div>"))
        else:
            header_html = \"\"\"
            <div style="display: grid; grid-template-columns: 40px 40px 200px 50px 50px 120px 60px 60px 60px 160px; gap: 5px; background: #2c3e50; color: white; padding: 10px; border-radius: 8px 8px 0 0; font-weight: bold; font-family: sans-serif; font-size: 11px; align-items: center;">
                <div style='text-align:center;'>Seç</div>
                <div style='text-align:center;'>ID</div>
                <div>URL / Metin</div>
                <div style='text-align:center;'>Ver</div>
                <div style='text-align:center;'>Lvl</div>
                <div>Görsel (Asset)</div>
                <div style='text-align:center;'>Color</div>
                <div style='text-align:center;'>Cont</div>
                <div style='text-align:center;'>Bright</div>
                <div style='text-align:center;'>İşlemler</div>
            </div>
            \"\"\"
            display(HTML(header_html))
            
            rows = []
            for i, row in df.iterrows():
                select_check = widgets.Checkbox(value=bool(row['selected']), layout={'width': '40px'})
                edit_btn = widgets.Button(description="", icon='pencil', button_style='info', layout={'width': '40px', 'height': '30px'})
                delete_btn = widgets.Button(description="", icon='trash', button_style='danger', layout={'width': '40px', 'height': '30px'})
                
                def on_selection_change(change, idx=i):
                    df_local = load_data()
                    df_local.at[idx, 'selected'] = change['new']
                    save_data(df_local)
                
                select_check.observe(on_selection_change, names='value')
                edit_btn.on_click(lambda b, idx=i: show_editor(idx))
                delete_btn.on_click(lambda b, idx=i: delete_row(idx))
                
                row_widget = widgets.HBox([
                    select_check,
                    widgets.Label(str(i+1), layout={'width': '40px'}, style={'text_align': 'center'}),
                    widgets.Label(str(row['words']), layout={'width': '200px'}),
                    widgets.Label(str(row['version']), layout={'width': '50px'}, style={'text_align': 'center'}),
                    widgets.Label(str(row['level']), layout={'width': '50px'}, style={'text_align': 'center'}),
                    widgets.Label(str(row['picture']) if pd.notna(row['picture']) and str(row['picture']) != 'nan' else "-", layout={'width': '120px'}),
                    widgets.Label("✅" if bool(row['colorized']) else "❌", layout={'width': '60px'}, style={'text_align': 'center'}),
                    widgets.Label(f"{float(row['contrast']):.1f}", layout={'width': '60px'}, style={'text_align': 'center'}),
                    widgets.Label(f"{float(row['brightness']):.1f}", layout={'width': '60px'}, style={'text_align': 'center'}),
                    widgets.HBox([edit_btn, delete_btn], layout={'width': '160px', 'gap': '5px', 'justify_content': 'center'})
                ], layout={'border-bottom': '1px solid #eee', 'padding': '2px 5px', 'align_items': 'center', 'font_family': 'sans-serif', 'font_size': '11px'})
                rows.append(row_widget)
            
            display(widgets.VBox(rows, layout={'border': '1px solid #eee', 'border-top': 'none', 'border-radius': '0 0 8px 8px', 'background': 'white', 'padding': '0'}))

def delete_row(idx):
    df = load_data()
    df = df.drop(idx).reset_index(drop=True)
    save_data(df)
    refresh_view()

def show_editor(idx=None):
    df = load_data()
    if idx is not None: item = df.iloc[idx]
    else: item = {'selected': True, 'words': '', 'version': 1, 'level': 'H', 'picture': '', 'colorized': True, 'contrast': 1.0, 'brightness': 1.0, 'save_name': ''}
    
    words_input = widgets.Text(value=str(item['words']), description='<b>URL/Metin:</b>', placeholder='https://...', style={'description_width': '100px'}, layout={'width': '100%'})
    picture_input = widgets.Text(value=str(item['picture']) if pd.notna(item['picture']) and str(item['picture']) != 'nan' else '', description='<b>Görsel Adı:</b>', placeholder='logo.png (Opsiyonel)', style={'description_width': '100px'}, layout={'width': '100%'})
    version_input = widgets.IntSlider(value=int(item['version']), min=1, max=40, step=1, description='Versiyon:', style={'description_width': '100px'})
    level_input = widgets.Dropdown(options=['L', 'M', 'Q', 'H'], value=str(item['level']), description='Hata Payı:', style={'description_width': '100px'})
    colorized_input = widgets.Checkbox(value=bool(item['colorized']), description='Renklendir (Colorized)', style={'description_width': '100px'})
    contrast_input = widgets.FloatSlider(value=float(item['contrast']), min=0.1, max=3.0, step=0.1, description='Kontrast:', style={'description_width': '100px'})
    brightness_input = widgets.FloatSlider(value=float(item['brightness']), min=0.1, max=3.0, step=0.1, description='Parlaklık:', style={'description_width': '100px'})
    save_name_input = widgets.Text(value=str(item['save_name']), description='<b>Kayıt Adı:</b>', placeholder='auto_generated.png', style={'description_width': '100px'}, layout={'width': '100%'})
    selected_input = widgets.Checkbox(value=bool(item.get('selected', True)), description='Seçili', style={'description_width': '100px'})

    save_btn = widgets.Button(description="Kaydet", button_style='primary', icon='check', layout={'width': '120px'})
    cancel_btn = widgets.Button(description="İptal", icon='times', layout={'width': '100px'})
    
    def on_save(b):
        new_row = {'selected': selected_input.value, 'words': words_input.value, 'version': version_input.value, 'level': level_input.value, 'picture': picture_input.value, 'colorized': colorized_input.value, 'contrast': contrast_input.value, 'brightness': brightness_input.value, 'save_name': save_name_input.value}
        current_df = load_data()
        if idx is not None:
            for col, val in new_row.items(): current_df.at[idx, col] = val
        else:
            current_df = pd.concat([current_df, pd.DataFrame([new_row])], ignore_index=True)
        save_data(current_df)
        refresh_view()

    save_btn.on_click(on_save)
    cancel_btn.on_click(lambda b: refresh_view())
    
    with output:
        clear_output()
        title = "✍️ Siparişi Düzenle" if idx is not None else "➕ Yeni Sipariş Ekle"
        display(HTML(f"<h3 style='color: #2980b9; margin-bottom: 20px;'>{title}</h3>"))
        form = widgets.VBox([selected_input, words_input, picture_input, widgets.HBox([version_input, level_input]), colorized_input, widgets.HBox([contrast_input, brightness_input]), save_name_input, widgets.HBox([save_btn, cancel_btn], layout={'margin': '30px 0 10px 0', 'gap': '15px'})], layout={'padding': '30px', 'border': '1px solid #3498db', 'border-radius': '15px', 'background': '#fdfdfd'})
        display(form)

def show_bulk_editor():
    df = load_data()
    if df.empty: return refresh_view()
    range_check = widgets.Checkbox(value=False, description='Belirli Satırlara Uygula (Range)')
    start_input = widgets.BoundedIntText(value=1, min=1, max=len(df), description='Başlangıç:')
    end_input = widgets.BoundedIntText(value=len(df), min=1, max=len(df), description='Bitiş:')
    version_check = widgets.Checkbox(value=False, description='Versiyonu Değiştir')
    version_input = widgets.IntSlider(value=40, min=1, max=40)
    level_check = widgets.Checkbox(value=False, description='Hata Payını Değiştir')
    level_input = widgets.Dropdown(options=['L', 'M', 'Q', 'H'], value='H')
    picture_check = widgets.Checkbox(value=False, description='Görseli Değiştir')
    picture_input = widgets.Text(value='', placeholder='logo.png')
    selected_check = widgets.Checkbox(value=False, description='Seçim Durumunu Değiştir')
    selected_input = widgets.Checkbox(value=True, description='Seçili Yap')
    apply_btn = widgets.Button(description="Seçilenlere Uygula", button_style='warning', icon='bolt')
    cancel_btn = widgets.Button(description="İptal", icon='times')

    def on_apply(b):
        current_df = load_data()
        start_idx = (start_input.value - 1) if range_check.value else 0
        end_idx = end_input.value if range_check.value else len(current_df)
        for i in range(max(0, start_idx), min(len(current_df), end_idx)):
            if version_check.value: current_df.at[i, 'version'] = version_input.value
            if level_check.value: current_df.at[i, 'level'] = level_input.value
            if picture_check.value: current_df.at[i, 'picture'] = picture_input.value
            if selected_check.value: current_df.at[i, 'selected'] = selected_input.value
        save_data(current_df)
        refresh_view()

    apply_btn.on_click(on_apply)
    cancel_btn.on_click(lambda b: refresh_view())
    with output:
        clear_output()
        display(HTML("<h3 style='color: #e67e22; margin-bottom: 20px;'>🛠️ Toplu Düzenleme Paneli</h3>"))
        display(widgets.VBox([widgets.VBox([range_check, widgets.HBox([start_input, end_input])]), widgets.HBox([version_check, version_input]), widgets.HBox([level_check, level_input]), widgets.HBox([picture_check, picture_input]), widgets.HBox([selected_check, selected_input]), widgets.HBox([apply_btn, cancel_btn])]))

refresh_view()
display(output)


## 🚀 4. Toplu İşlemi Başlat


In [ ]:
import sys
import time
from IPython.display import clear_output

print("🚀 İşlem başlatılıyor...")
!python batch_process.py
print("\n✅ İşlem tamamlandı!")


## ✨ 5. Sonuçları Önizle


In [ ]:
import os
import base64
from IPython.display import display, HTML

output_dir = 'output'
if os.path.exists(output_dir):
    files_list = sorted([f for f in os.listdir(output_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif'))])
    if not files_list:
        display(HTML("<div style='color: red; font-weight: bold;'>❌ Üretilen dosya bulunamadı.</div>"))
    else:
        display(HTML("<h2 style='color: #2c3e50; text-align: center;'>✨ Oluşturulan QR Kodlar</h2>"))
        html_content = '<div style="display: flex; flex-wrap: wrap; gap: 20px; justify-content: center; padding: 20px; background: #f8f9fa; border-radius: 15px;">'
        for f in files_list:
            f_path = os.path.join(output_dir, f)
            with open(f_path, "rb") as img_file:
                encoded_string = base64.b64encode(img_file.read()).decode('utf-8')
            mime_type = "image/gif" if f.lower().endswith(".gif") else "image/png"
            html_content += f'''
            <div style="text-align: center; background: white; padding: 10px; border-radius: 12px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); width: 200px;">
                <img src="data:{mime_type};base64,{encoded_string}" style="width: 180px; height: 180px; object-fit: contain; border-radius: 8px;">
                <p style="font-size: 12px; margin-top: 8px; color: #555; word-break: break-all;">{f}</p>
            </div>
            '''
        html_content += '</div>'
        display(HTML(html_content))


## 🔍 6. QR Kod Çalışma Testi (Scannability Test)


In [ ]:
import cv2
import os
from IPython.display import display, HTML

output_dir = 'output'
detector = cv2.QRCodeDetector()

print("🔍 QR kodlar test ediliyor...")
results = []
if os.path.exists(output_dir):
    for f in sorted(os.listdir(output_dir)):
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            img = cv2.imread(os.path.join(output_dir, f))
            data, bbox, _ = detector.detectAndDecode(img)
            status = "✅ Başarılı" if data else "❌ Okunamadı"
            results.append((f, data if data else "-", status))

if results:
    html = "<table style='width: 100%; border-collapse: collapse; margin-top: 10px;'>"
    html += "<tr style='background: #2c3e50; color: white;'><th style='padding: 10px;'>Dosya</th><th style='padding: 10px;'>İçerik</th><th style='padding: 10px;'>Durum</th></tr>"
    for f, d, s in results:
        color = "#27ae60" if "Başarılı" in s else "#e74c3c"
        html += f"<tr style='border-bottom: 1px solid #ddd;'><td style='padding: 8px;'>{f}</td><td style='padding: 8px;'>{d}</td><td style='padding: 8px; color: {color}; font-weight: bold;'>{s}</td></tr>"
    html += "</table>"
    display(HTML(html))
else:
    print("Test edilecek statik görsel bulunamadı.")


## 📥 7. Sonuçları İndir (ZIP)


In [ ]:
import shutil
from google.colab import files
import datetime

now = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f'amazing_qr_results_{now}'

try:
    shutil.make_archive(zip_name, 'zip', 'output')
    files.download(f'{zip_name}.zip')
    display(HTML(f"<div style='padding: 20px; background: #d4edda; color: #155724; border-radius: 8px;'>✅ <b>{zip_name}.zip</b> oluşturuldu ve indiriliyor...</div>"))
except Exception as e:
    print(f"❌ Hata: {e}")
